# v2.01 (proposal-accurate Multi-Scale PatchTST) + faithful PatchTST baseline — validation only

**v2.01** = the committee-slide architecture with every number chosen by Siwat (2026-10-19), in `configs/v2_01.json`:
18 inputs, look-back 126, patches 1 / 5 / 21 (day / week / month, no overlap), RevIN on, PatchTST's BatchNorm encoder
(d_model 128, 16 heads, 3 layers), attention scale fusion, stock + sector embedding, two heads trained on trend **and** price (1 : 1).

**Baseline** = PatchTST as in Nie et al. (2023) and its official code (`configs/baseline_patchtst.json`). It runs only once its
10 model numbers are filled in; until then step 6 skips it.

**Runtime: A100 GPU** (Siwat, 2026-10-19). Batch 256 at these sizes is estimated at 25–60 GB; a 40 GB A100 may not be enough and an
80 GB one should be. Each epoch prints the GPU peak; if memory runs out, the run stops with the options. Walk-forward: off (Siwat).
Outputs: `MyDrive/Senior/results/v2xx/`. If Colab disconnects, Run all again: finished runs are skipped. Test block stays sealed.

In [ ]:
!nvidia-smi -L
import torch
name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none'
mem = torch.cuda.get_device_properties(0).total_memory / 2**30 if torch.cuda.is_available() else 0
print(f'GPU: {name}, {mem:.0f} GB')
assert 'A100' in name, 'stop: switch the runtime to an A100 GPU (Runtime -> Change runtime type)'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SENIOR = '/content/drive/MyDrive/Senior'
PRICE_DIR, OUT = f'{SENIOR}/prices', f'{SENIOR}/results/v2xx'
RUN_WALK_FORWARD = False   # Siwat, 2026-10-19: official window only
import os; os.makedirs(OUT, exist_ok=True)

In [ ]:
!rm -rf /content/Senior-Project && git clone -q https://github.com/Smith8bit/Senior-Project.git /content/Senior-Project
%cd /content/Senior-Project
!git log --oneline -1
import hashlib, json, subprocess, sys
from pathlib import Path
for f in ['scripts/train_v2xx.py', 'src/proposal_model.py', 'src/patchtst_faithful.py', 'src/encoders.py', 'src/features_proposal.py',
          'src/run_config.py', 'configs/v2_01.json', 'configs/baseline_patchtst.json', 'tests/test_v2xx.py', 'tests/test_v2xx_pipeline.py', 'scripts/plot_v2xx_training.py']:
    assert os.path.exists(f), f'stop: push the v2.01 code first ({f} missing)'
    print(f, hashlib.sha256(open(f, 'rb').read()).hexdigest()[:12])

In [ ]:
!mkdir -p data/raw/prices/siamchart data/raw/prices/yahoo
!cp "$PRICE_DIR/set50_union_2019_2025.csv" data/raw/prices/siamchart/
!cp "$PRICE_DIR/yahoo_events_2019_2025.csv" data/raw/prices/yahoo/
expected = {'data/raw/prices/siamchart/set50_union_2019_2025.csv': '9d7ce4c3e72a974979566dc089d5b8819af54dc24f62b71240f5b4716b556e6f',
            'data/raw/prices/yahoo/yahoo_events_2019_2025.csv': 'd021a58df95fa487f1e49eb50617f6b3187b025b6377235c6180166b1ec84d53',
            'data/derived/sentiment_scores.csv': '7369fd981c4ccffc017d6d5de50ed95c6a4ac4d843bbc6053366029ce614bb1b'}
for path, h in expected.items():
    got = hashlib.sha256(open(path, 'rb').read()).hexdigest()
    print('OK ' if got == h else 'MISMATCH ', path)
    assert got == h, 'stop: a file differs from the PC copy'

In [ ]:
# 3. Configs: v2.01 must be complete; the baseline runs only when complete
sys.path.insert(0, 'src')
import run_config as RC
cfg = RC.load('configs/v2_01.json')
print('v2.01 ready:', json.dumps({k: cfg[k] for k in ('model', 'loss', 'training')}, indent=1))
base_missing = RC.problems(json.load(open('configs/baseline_patchtst.json')))
RUN_BASELINE = not base_missing
print('baseline ready' if RUN_BASELINE else 'baseline skipped, undecided:\n  ' + '\n  '.join(base_missing))

In [ ]:
# 4. Tests, including the full-pipeline scramble test (about 5 minutes); the notebook stops if any fails
for t in ['tests/test_v2xx.py', 'tests/test_v2xx_pipeline.py', 'tests/test_pipeline.py']:
    r = subprocess.run(['python', t], capture_output=True, text=True)
    print(f'--- {t}'); print('\n'.join(l for l in r.stdout.splitlines() if 'defined on' not in l)[-3000:])
    assert r.returncode == 0 and 'FAIL' not in r.stdout, f'stop: {t} failed\n' + r.stderr[-2000:]

In [ ]:
# 5. v2.01, official validation window, 3 seeds
def done(path, n=3):
    try:
        r = json.load(open(path)); return len(r['runs']) == n and 'summary' in r
    except Exception:
        return False
def run(label, config, out, extra=()):
    if done(out):
        print(f'=== {label}: already done'); return
    print(f'=== {label}'); sys.stdout.flush()
    # stream the script's output into this cell (plain subprocess.run output is not shown in Colab)
    p = subprocess.Popen(['python', '-u', 'scripts/train_v2xx.py', '--config', config, '--out', out, *extra],
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='')
    if p.wait() != 0:
        raise RuntimeError(f'{label} failed (exit {p.returncode}); the error is printed just above')
V201 = f'{OUT}/v2_01.json'
run('v2.01 official window', 'configs/v2_01.json', V201)

In [ ]:
# 6. Faithful PatchTST baseline (only if its numbers are filled in)
BASE = f'{OUT}/baseline_patchtst.json'
if RUN_BASELINE:
    run('faithful PatchTST, official window', 'configs/baseline_patchtst.json', BASE)
else:
    print('baseline skipped (numbers not decided yet)')

In [ ]:
# 7. Summary beside v2 and the floors (validation macro-F1; price R2_OS and baht errors)
V2 = {'h1': 0.387, 'h5': 0.396, 'h10': 0.378}; BAR = {'h1': 0.393, 'h5': 0.400, 'h10': 0.407}
def show(path, label):
    r = json.load(open(path)); s, fl = r['summary']['model'], r['floors']
    print(f"--- {label}")
    print(f"{'':6}{'F1':>16}{'v2':>7}{'bar':>7}{'continues':>11}{'reverses':>10}{'R2_OS':>10}{'RMSE baht':>11}{'persist.':>10}")
    for h in ('h1', 'h5', 'h10'):
        m = s[h]
        print(f"{h:6}{m['macro_f1']['mean']:>9.3f} ± {m['macro_f1']['std']:.3f}{V2[h]:>7.3f}{BAR[h]:>7.3f}"
              f"{fl[h]['trend_continues']['macro_f1']:>11.3f}{fl[h]['trend_reverses']['macro_f1']:>10.3f}"
              f"{m['r2_os_vs_zero']['mean']:>+10.4f}{m['price_rmse_baht']['mean']:>11.3f}{fl[h]['persistence_price_baht']['rmse']:>10.3f}")
    for x in r['runs']:
        sw = [v for k, v in x.items() if k.startswith('scale_weights')]
        print('seed', x['seed'], 'epochs', x['epochs_run'], 'used', x['best_epoch'], 'params', x['params']['total'],
              'GPU peak GB', x['history'][-1].get('gpu_peak_gb'), 'scale weights', sw[0]['overall'] if sw else '-')
show(V201, 'v2.01')
if RUN_BASELINE and done(BASE):
    show(BASE, 'faithful PatchTST')

In [ ]:
# 8. Learning curves: training loss by task, inner hold-out macro-F1 and price RMSE per epoch, kept epoch marked.
# Saved beside the results on Drive (v2_01_training.png, baseline_patchtst_training.png).
from IPython.display import Image, display
sys.path.insert(0, 'scripts')
import importlib, plot_v2xx_training as PT
importlib.reload(PT)
for path in [V201] + ([BASE] if RUN_BASELINE and done(BASE) else []):
    png = PT.plot(Path(path))
    print('saved', png)
    display(Image(filename=str(png)))